# Memory mapping dataset
hi so like loading it in ram ate up over 300 gb of ram so i gotta make a memory map :3

In [1]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
import os
import time
import numpy as np
import tokenizer_fast

INPUT_FILE = "output.txt"
OUTPUT_BIN = "dataset.bin"
CHUNK_SIZE_BYTES = 2_000_000  # 2 MB chunks for fast feedback & optimal worker scaling
VOCAB_FILE = "vocab_id.vocab"

try:
    mp.set_start_method("fork", force=True)
except RuntimeError:
    pass

print(f"Loading vocabulary from '{VOCAB_FILE}'...")
vocab_to_id, _, bpe_ranks = tokenizer_fast.read_vocab_dict(VOCAB_FILE)
print(f"Loaded {len(vocab_to_id):,} vocab tokens.")


def get_chunk_offsets(file_path, chunk_size):
    file_size = os.path.getsize(file_path)
    offsets = []
    with open(file_path, "rb") as f:
        start = 0
        while start < file_size:
            f.seek(start + chunk_size)
            f.readline()
            end = min(f.tell(), file_size)
            offsets.append((start, end - start))
            start = end
    return offsets, file_size


def process_chunk_from_file(args):
    chunk_idx, offset, length = args

    with open(INPUT_FILE, "r", encoding="utf-8", errors="ignore") as f:
        f.seek(offset)
        chunk = f.read(length)

    token_ids = tokenizer_fast.encode(chunk, vocab_to_id, bpe_ranks)
    np_tokens = np.array(token_ids, dtype=np.uint32)

    return chunk_idx, np_tokens, len(chunk)


# if __name__ == "__main__":
#     num_workers = mp.cpu_count()
#     print(f"Calculating chunk boundaries for 60GB file '{INPUT_FILE}'...")
#
#     start_time = time.perf_counter()
#     chunk_offsets, total_file_bytes = get_chunk_offsets(INPUT_FILE, CHUNK_SIZE_BYTES)
#     total_chunks = len(chunk_offsets)
#
#     print(f"Split '{INPUT_FILE}' ({total_file_bytes / (1024**3):.2f} GB) into {total_chunks:,} chunks.")
#     print(f"Pre-tokenizing using {num_workers} processes...")
#
#     tasks = [(idx, offset, length) for idx, (offset, length) in enumerate(chunk_offsets)]
#
#     total_tokens = 0
#     total_chars = 0
#     processed_chunks = 0
#
#     token_buffers = [None] * total_chunks
#
#     with ProcessPoolExecutor(max_workers=num_workers) as executor:
#         futures = {executor.submit(process_chunk_from_file, task): task[0] for task in tasks}
#
#         for fut in as_completed(futures):
#             chunk_idx, np_tokens, char_count = fut.result()
#             token_buffers[chunk_idx] = np_tokens
#
#             total_tokens += len(np_tokens)
#             total_chars += char_count
#             processed_chunks += 1
#
#             elapsed = time.perf_counter() - start_time
#             tok_per_sec = total_tokens / elapsed
#             mb_per_sec = (total_chars / (1024**2)) / elapsed
#
#             print(
#                 f"\rProgress: {processed_chunks}/{total_chunks} ({processed_chunks/total_chunks*100:.1f}%) | "
#                 f"Tokens: {total_tokens:,} | Throughput: {tok_per_sec:,.0f} tok/s ({mb_per_sec:.2f} MB/s)",
#                 end="",
#                 flush=True,
#             )
#
#     print("\n\nWriting tokenized dataset directly to binary disk format...")
#     with open(OUTPUT_BIN, "wb") as f_out:
#         for buf in token_buffers:
#             f_out.write(buf.tobytes())
#
#     total_time = time.perf_counter() - start_time
#     avg_tok_per_sec = total_tokens / total_time
#     avg_mb_per_sec = (total_file_bytes / (1024**2)) / total_time
#
#     print("\n" + "=" * 50)
#     print("FINISHED PRE-TOKENIZATION")
#     print("=" * 50)
#     print(f"Total Tokens Saved : {total_tokens:,}")
#     print(f"Total Time Taken    : {total_time:.2f}s")
#     print(f"Average Throughput  : {avg_tok_per_sec:,.0f} tokens/sec ({avg_mb_per_sec:.2f} MB/s)")
#     print(f"Saved binary file to '{OUTPUT_BIN}' ({os.path.getsize(OUTPUT_BIN) / (1024**3):.2f} GB)")

Loading vocabulary from 'vocab_id.vocab'...
Loaded 127,648 vocab tokens.


In [ ]:
%%writefile ddp_worker.py

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from transformers import get_cosine_schedule_with_warmup
import torch.distributed as dist
import numpy as np
import os
import time
import tokenizer
from contextlib import nullcontext

vocab_to_id, id_to_vocab, bpe_ranks = tokenizer.read_vocab_dict("vocab_id.vocab")

# def setup_ddp():
#     """Initializes Distributed Data Parallel (DDP) environment."""
#     dist.init_process_group(backend="nccl")
#     local_rank = int(os.environ["LOCAL_RANK"])
#     torch.cuda.set_device(local_rank)
#     return local_rank

class MemMapDataset(Dataset):
    def __init__(self, bin_file_path, seq_len=1024):
        file_size = os.path.getsize(bin_file_path)
        num_tokens = file_size // 4
        self.tokens = torch.from_file(
            bin_file_path,
            shared=True,
            size=num_tokens,
            dtype=torch.int32
        )

        # self.tokens = np.memmap(bin_file_path, dtype=np.uint32, mode='r')
        self.seq_len = seq_len

    def __len__(self):
        return (len(self.tokens) - 1) // self.seq_len

    def __getitem__(self, idx):
        start_idx = idx * self.seq_len
        end_idx = start_idx + self.seq_len

        x = self.tokens[start_idx:end_idx].clone().long()
        y = self.tokens[start_idx+1:end_idx+1].clone().long()
        return x, y
class TextDataset(Dataset):
    def __init__(self, raw_text, tokenizer, vocab_to_id, bpe_ranks, seq_len = 128):
        print("Tokenizing dataset...")
        self.tokens = torch.tensor(
            tokenizer.encode(raw_text, vocab_to_id, bpe_ranks),
            dtype=torch.int32
        )
        self.seq_len = seq_len
        print(f"Total tokens in dataset: {len(self.tokens):,}")
    def __len__(self,):
        return (len(self.tokens)-1)//self.seq_len

    def __getitem__(self, idx):
        start_idx = idx*self.seq_len
        end_idx = start_idx + self.seq_len
        x = self.tokens[start_idx:end_idx]
        y= self.tokens[start_idx+1:end_idx+1]
        return x ,y

class TransformerLayer(nn.Module):
    def __init__(self, d_model=768, heads=12):
        super().__init__()
        self.heads = heads
        self.d_model = d_model
        self.d_k = d_model//heads

        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

        self.ffl1 = nn.Linear(d_model, 4*d_model, bias=False)
        self.ffl2 = nn.Linear(4*d_model, d_model, bias=False)
        self.GELU = nn.GELU()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)


    def _ffn(self, x):
        return self.ffl2(self.GELU(self.ffl1(x)))
    def _attention(self, Q, K, V):
        seq_len = Q.shape[-2]
        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)

        mask = torch.triu(
            torch.full((seq_len, seq_len), float('-inf'), device=Q.device),
            diagonal=1
        )
        masked_scores = scores+mask
        return nn.functional.softmax(masked_scores, dim=-1) @ V
    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        norm_x = self.ln1(x)
        Q = self.w_q(norm_x)
        K = self.w_k(norm_x)
        V = self.w_v(norm_x)

        Q = Q.view(batch_size, seq_len, self.heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.heads, self.d_k).transpose(1, 2)

        attn_weights = torch.nn.functional.scaled_dot_product_attention(
            Q, K, V, is_causal=True
        )
        # attn_weights = self._attention(Q, K, V)

        out = attn_weights.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        x = x + self.w_o(out)

        x = x + self._ffn(self.ln2(x))

        return x


class GPT(nn.Module):
    def __init__(self, vocab_size=len(vocab_to_id), d_model=768, max_seq_len=1024, heads=12, num_layers=4):
        super().__init__()
        self.max_seq_len=max_seq_len
        self.w_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)
        self.layers = nn.ModuleList([
            TransformerLayer(d_model=d_model, heads=heads) for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.w_embedding.weight = self.lm_head.weight
    def forward(self, input_tokens):
        batch_size, seq_len = input_tokens.shape
        # input_tokens = input_tokens.long()
        # seq_len = input_tokens.shape[0]
        positions = torch.arange(seq_len, device=input_tokens.device)
        x = self.w_embedding(input_tokens) + self.pos_embedding(positions)
        for layer in self.layers:
            x = layer(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits

class SkipSampler(torch.utils.data.Sampler):
    def __init__(self, base, skip):
        self.base, self.skip = base, skip
    def __iter__(self):
        it = iter(self.base)
        for _ in range(self.skip):
            next(it)
        return it
    def __len__(self):
        return len(self.base) - self.skip

@torch.no_grad()
def generate(model, prompt_ids, max_new_tokens=50, temperature=1.0):
    model.eval()
    if prompt_ids.ndim == 1:
        prompt_ids = prompt_ids.unsqueeze(0)
    generated = prompt_ids.clone()

    for _ in range(max_new_tokens):
        input_tokens = generated[:, -model.max_seq_len:]
        logits = model(input_tokens)
        next_token = logits[:, -1, :] / temperature

        probs = nn.functional.softmax(next_token, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        generated = torch.cat([generated, next_token], dim=1)
    return generated.squeeze(0)

@torch.no_grad()
def generate_sample_text(model, prompt_text="The ", max_new_tokens=50, device='cuda'):
    model.eval()
    input_ids = tokenizer.encode(prompt_text, vocab_to_id, bpe_ranks)
    x = torch.tensor([input_ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        x_cond = x[:, -1024:] # Context window clip
        logits = model(x_cond)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        x = torch.cat((x, next_token), dim=1)

    model.train()
    return tokenizer.decode(x[0].tolist(), id_to_vocab)

def format_time(seconds):
    """Converts seconds to HH:MM:SS format."""
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:02d}:{m:02d}:{s:02d}" if h > 0 else f"{m:02d}:{s:02d}"

def setup_ddp(local_rank, world_size):
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    torch.cuda.set_device(local_rank)

    dist.init_process_group("nccl")

def cleanup_ddp():
    dist.destroy_process_group()

@torch.no_grad()
def generate_sample(model, prompt_tokens, max_new_tokens=50, temperature=0.8, top_k=40):
    model.eval()
    raw_model = model.module if hasattr(model, 'module') else model
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    x = prompt_tokens.clone().unsqueeze(0) # (1, seq_len)

    for _ in range(max_new_tokens):
        x_cond = x[:, -raw_model.max_seq_len:]

        with torch.amp.autocast('cuda', dtype=dtype):
            logits = model(x_cond)

        logits = logits[:, -1, :] / temperature

        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')

        probs = torch.nn.functional.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        x = torch.cat((x, next_token), dim=1)

    model.train()
    return x[0]

def train_worker(local_rank, world_size):
    setup_ddp(local_rank, world_size)
    device = torch.device(f"cuda:{local_rank}")
    is_main_process = (local_rank == 0)

    torch.cuda.empty_cache()
    ckpt = torch.load("checkpoint_step_v2_XXXXX.pt", map_location=device)
    start_step = ckpt["step"] + 1

    BATCH_SIZE_PER_GPU = 10
    GRAD_ACCUM_STEPS = 6
    SEQ_LEN = 1024
    EPOCHS = 5
    LEARNING_RATE = 3e-4
    LOG_INTERVAL = 50
    dataset = MemMapDataset("dataset.bin", seq_len=SEQ_LEN)

    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=local_rank, shuffle=True)
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE_PER_GPU,
        sampler=SkipSampler(sampler, start_step * BATCH_SIZE_PER_GPU),
        num_workers=3,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
    )
    raw_model = GPT(
        vocab_size=len(vocab_to_id),
        max_seq_len=SEQ_LEN,
        d_model=768,
        heads=12,
        num_layers=6
    ).to(device)

    optimizer = torch.optim.AdamW(raw_model.parameters(), lr=LEARNING_RATE)
    WARMUP_STEPS = 2000  # Ramps LR from 0 to 3e-4 over first 2000 steps
    TOTAL_OPTIMIZER_STEPS = (len(dataloader) // GRAD_ACCUM_STEPS) * EPOCHS

    scheduler = get_cosine_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=TOTAL_OPTIMIZER_STEPS,
        # min_lr_ratio=0.1 # Decays down to 10% of peak LR (3e-5) at the end
    )

    raw_model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])

    model = torch.compile(raw_model)

    model = DDP(model, device_ids=[local_rank])


    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=(dtype == torch.float16))
    loss_fn = nn.CrossEntropyLoss()

    total_steps = len(dataloader) + start_step
    start_training_time = time.time()

    model.train()
    for epoch in range(EPOCHS):
        sampler.set_epoch(epoch)
        total_loss = 0.0
        epoch_start_time = time.time()
        step_start_time = time.time()
        optimizer.zero_grad(set_to_none = True)
        if is_main_process:
            log_f = open("metrics.csv", "a", buffering=1)
            if log_f.tell() == 0:
                log_f.write("micro_step,opt_step,tokens,loss_avg,grad_norm_last,grad_norm_max,lr,tok_per_s,vram_gb\n")
        last_gn, max_gn = float("nan"), 0.0
        for step, (x_batch, y_batch) in enumerate(dataloader, start=start_step):
            x_batch = x_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)

            is_accumulating = (step + 1) % GRAD_ACCUM_STEPS != 0
            context = model.no_sync() if is_accumulating else nullcontext()
            with context:
                with torch.amp.autocast('cuda', dtype=dtype):
                    logits = model(x_batch)
                    unscaled_loss = loss_fn(
                        logits.view(-1, len(vocab_to_id)),
                        y_batch.view(-1)
                    )
                    loss = unscaled_loss / GRAD_ACCUM_STEPS

                if dtype == torch.float16:
                    scaler.scale(loss).backward()
                else:
                    loss.backward()

            total_loss += unscaled_loss.item()
            # * GRAD_ACCUM_STEPS
            if not is_accumulating:
                if dtype == torch.float16:
                    scaler.unscale_(optimizer)
                    gn = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0).item()
                    last_gn, max_gn = gn, max(max_gn, gn)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    gn = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0).item()
                    last_gn, max_gn = gn, max(max_gn, gn)
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)


            if is_main_process and (step + 1) % 50 == 0:
                elapsed_interval = time.time() - step_start_time
                steps_per_sec = LOG_INTERVAL / elapsed_interval
                tokens_per_sec = steps_per_sec * (BATCH_SIZE_PER_GPU * world_size) * SEQ_LEN

                steps_remaining = total_steps - (step + 1)
                eta_epoch = steps_remaining / steps_per_sec if steps_per_sec > 0 else 0

                vram_info = ""
                if torch.cuda.is_available():
                    vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
                    vram_info = f" | Peak VRAM: {vram_mb:.0f}MB"
                print(
                    f"Epoch [{epoch+1}/{EPOCHS}] | "
                    f"Step [{step+1}/{total_steps}] | "
                    f"Loss: {unscaled_loss.item():.4f} | "
                    f"Speed: {steps_per_sec:.2f} st/s ({tokens_per_sec:.0f} tok/s) | "
                    f"ETA Epoch: {format_time(eta_epoch)}"
                    f"{vram_info}"
                )
                step_start_time = time.time()

                predicted_ids = logits[0][:-1].argmax(dim=-1).tolist()
                target_text_clean = tokenizer.decode(y_batch[0].tolist(), id_to_vocab).replace('\n', ' ')
                sample_pred_clean = tokenizer.decode(predicted_ids, id_to_vocab).replace('\n', ' ')

                print(f"TARGET BATCH SAMPLE :\n\"{target_text_clean[:150]}...\"")
                print(f"MODEL PREDICTION    :\n\"{sample_pred_clean[:150]}...\"\n")
            if is_main_process and (step + 1) % 500 == 0:
                print(f"SAMPLE AUTOREGRESSIVE STRING: \n\"{generate_sample_text(model).replace('\n', ' ')}\"")

            if is_main_process and (step+1) % 5000 == 0:
                checkpoint = {
                    'model': model.module.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'epoch': epoch,
                    'step': step,
                }
                torch.save(checkpoint, f"checkpoint_step_{step+1}.pt")
        if is_main_process:
            epoch_time = time.time() - epoch_start_time
            avg_loss = total_loss / len(dataloader)
            print(f"--- Epoch {epoch+1} Complete | Average Loss: {avg_loss:.4f} ---\n")
    if is_main_process:
        total_training_time = time.time() - start_training_time
        print(f"🎉 Training Completed in {format_time(total_training_time)}!")
    cleanup_ddp()

if __name__ == "__main__":
    # torchrun automatically handles rank and world_size environment variables
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    train_worker(local_rank, world_size)


In [3]:
!torchrun --nproc_per_node=2 ddp_worker.py

W0925 20:26:08.542000 37720 .venv/lib/python3.12/site-packages/torch/distributed/run.py:982] 
W0925 20:26:08.542000 37720 .venv/lib/python3.12/site-packages/torch/distributed/run.py:982] *****************************************
W0925 20:26:08.542000 37720 .venv/lib/python3.12/site-packages/torch/distributed/run.py:982] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0925 20:26:08.542000 37720 .venv/lib/python3.12/site-packages/torch/distributed/run.py:982] *****************************************
Epoch [1/5] | Step [50/720161] | Loss: 9.1813 | Speed: 2.34 st/s (47940 tok/s) | ETA Epoch: 85:27:09 | Peak VRAM: 19517MB
TARGET BATCH SAMPLE :
" to other cities, that they are not only passing the tax they should be paying onto their customer, they are actually willing to threaten working men ..."
MODEL PREDICTION    :
"          

import torch.multiprocessing as mp

if __name__ == "__main__":
    # Force clean process creation instead of fork
    try:
        mp.set_start_method("spawn", force=True)
    except RuntimeError:
        pass

    world_size = 2
    mp.spawn(train_worker, args=(world_size,), nprocs=world_size, join=True)